# Data exploration

In [12]:
from src.lexicon_absa import LexiconABSA
from src.transformer_absa import TransformerABSA
from src.llm_absa import LLMABSA
import json

## LexiconABSA Exploration
### Conceptual Overview
The LexiconABSA model is a rule-based approach that uses linguistic parsing and sentiment lexicons (e.g., VADER) to identify aspects and assign sentiment polarities.
It relies on dependency parsing and fixed dictionaries, making it fast and interpretable.
However, it can miss implicit sentiments or complex relationships between words (e.g. sarcasm).

In [13]:
file_path = '../data/test_samples_small.json'
with open(file_path, 'r') as f:
    sample_texts = json.load(f)

lexicon_model = LexiconABSA()

for text in sample_texts:
    print(f"\n🟩 Text: {text}")
    print("→", lexicon_model.analyze(text))


🟩 Text: The â€˜friendlyâ€™ waiter ignored us the whole night â€” amazing experience!
→ [AspectSentiment(aspect='waiter', sentiment='negative', confidence=0.398472314, text_span=(0, 25)), AspectSentiment(aspect='us', sentiment='negative', confidence=0.3182, text_span=(34, 36))]

🟩 Text: I loved the hotel location, but the room was small.
→ [AspectSentiment(aspect='room', sentiment='neutral', confidence=0.0, text_span=(32, 40)), AspectSentiment(aspect='i', sentiment='positive', confidence=0.5994, text_span=(0, 1)), AspectSentiment(aspect='hotel location', sentiment='positive', confidence=0.5994, text_span=(8, 26))]

🟩 Text: The pizza was cold and the waiter was rude.
→ [AspectSentiment(aspect='pizza', sentiment='neutral', confidence=0.0, text_span=(0, 9)), AspectSentiment(aspect='waiter', sentiment='negative', confidence=0.4588, text_span=(23, 33))]

🟩 Text: The ambience was nice and the prices were fair.
→ [AspectSentiment(aspect='ambience', sentiment='positive', confidence=0.4215, tex

### Observations

Correctly handles contextual negations (“not good” → negative).

Sometimes it misses the aspect and doesn't catch it at all if it's implicit(‘signature cocktail’ - name)

The fastest model out of all

Doesn't catch sarcasm

Sometimes doesn't catch the sentiment because the vocabulary for sentiment is too narrow (marks stale as neutral)

## TransformerABSA Exploration
### Conceptual Overview

The TransformerABSA model uses contextual embeddings from transformer architectures (e.g., RoBERTa, DeBERTa) to detect aspect terms and infer their sentiment.
Unlike rule-based methods, it learns complex language relationships and handles sarcasm, and long dependencies.
However, it requires more computational resources and is less interpretable.

In [8]:
transformer_model = TransformerABSA()

for text in sample_texts:
    print(f"\n🟦 Text: {text}")
    print("→", transformer_model.analyze(text))

Device set to use cpu



🟦 Text: The ‘friendly’ waiter ignored us the whole night — amazing experience!
→ [('waiter', 'negative')]

🟦 Text: I loved the hotel location, but the room was small.
→ [('hotel', 'positive'), ('room', 'negative')]

🟦 Text: The pizza was cold and the waiter was rude.
→ [('pizza', 'negative'), ('waiter', 'negative')]

🟦 Text: The ambience was nice and the prices were fair.
→ [('ambience', 'positive'), ('prices', 'positive')]

🟦 Text: The coffee was too bitter but strong.
→ [('coffee', 'negative')]

🟦 Text: The burger was juicy, but the bun was stale and the fries were soggy.
→ [('burger', 'positive'), ('bun', 'negative'), ('fries', 'negative')]

🟦 Text: They finally fixed the air conditioning — about time!
→ [('air conditioning', 'neutral')]

🟦 Text: Their ‘signature cocktail’ was just juice with a fancy name.
→ [('cocktail', 'positive'), ('juice', 'neutral')]

🟦 Text: Absolutely loved the atmosphere, but the food was a disaster!
→ [('atmosphere', 'positive'), ('food', 'negative')]


### Observations

Correctly handles contextual negations as well

More accurate sentiment detection but may split some phrases(hotel location - hotel).

A bit slower than lexicon model (~0.3–0.8 s per text).

Catches sarcasm (waiter - negative)

The vocabulary is wide enough to catch the sentiment of mild-difficult cases (disaster - negative)
But doesn't catch implicit difficult cases (Their ‘signature cocktail’ was just juice with a fancy name. ('cocktail', 'positive'), ('juice', 'neutral'))

LLMABSA Exploration
### Conceptual Overview

The LLMABSA implementation uses a large language model (e.g., LLaMA 3) prompted to perform aspect-based sentiment analysis in natural language.
It’s zero-shot or few-shot — meaning it doesn’t rely on fine-tuning but on prompt design.
It’s capable of handling complex, nuanced sentences but may hallucinate non-existent aspects or vary in JSON structure.

In [18]:
llm_model = LLMABSA()

for text in sample_texts:
    print(f"\n🟥 Text: {text}")
    print("→", llm_model.analyze(text))


🟥 Text: The â€˜friendlyâ€™ waiter ignored us the whole night â€” amazing experience!
→ [AspectSentiment(aspect='waiter', sentiment='negative', confidence=0.95, text_span=(13, 18)), AspectSentiment(aspect='experience', sentiment='positive', confidence=0.93, text_span=(25, 36))]

🟥 Text: I loved the hotel location, but the room was small.
→ [AspectSentiment(aspect='hotel location', sentiment='positive', confidence=0.96, text_span=(5, 13)), AspectSentiment(aspect='room', sentiment='negative', confidence=0.91, text_span=(18, 22))]

🟥 Text: The pizza was cold and the waiter was rude.
→ [AspectSentiment(aspect='pizza', sentiment='negative', confidence=0.87, text_span=(4, 9)), AspectSentiment(aspect='waiter', sentiment='negative', confidence=0.95, text_span=(19, 25))]

🟥 Text: The ambience was nice and the prices were fair.
→ [AspectSentiment(aspect='ambience', sentiment='positive', confidence=0.87, text_span=(1, 8)), AspectSentiment(aspect='prices', sentiment='positive', confidence=0.82, te

### Observations

Demonstrates strong contextual understanding and nuanced sentiment interpretation.

It correctly identifies multiple aspects per sentence,

Distinguishes between contrasting sentiments,

Handles mild implicit sentiment with impressive consistency.

However, it occasionally duplicates or overgeneralizes aspects, and it struggles with subtle sarcasm or overlapping polarities on the same entity.